
##### Read CSV File from Azure Data Lake Storage Account
 CSV Source File Path : "abfss://working-labs@datalakestorageaccountname.dfs.core.windows.net/bronze/daily-pricing/csv"

JSON  File Path : "abfss://working-labs@datalakestorageaccountname.dfs.core.windows.net/bronze/daily-pricing/json"


##### Spark Methods
- <a href="https://spark.apache.org/docs/latest/sql-getting-started.html#starting-point-sparksession" target="_blank">SparkSession</a>

- <a href="https://spark.apache.org/docs/latest/sql-data-sources-load-save-functions.html" target="_blank">GenericDataFrameReader</a>: **`json`**,**`csv`**,  **`option (header,inferSchema)`** ,  **`schema`**


##### DateTime Methods

**`current_timestamp()`** records the timestamp when the code is executed

- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/column.html" target="_blank">ColumnFunctions</a>: **`cast`**
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#datetime-functions" target="_blank">Built-In DateTime Functions</a>: **`date_format`**, **`to_date`**,**`year`**, **`month`**, **`dayofmonth`**, **`minute`**, **`second`**

##### Date Time Functions

| Method | Description |
| --- | --- |
| **`current_timestamp`** | Returns the current timestamp at the start of query evaluation as a timestamp column |
| **`date_format`** | Converts a date/timestamp/string to a value of string in the format specified by the date format given by the second argument. |
| **`dayofmonth`** | Extracts the day of the month as an integer from a given date/timestamp/string |


#####  Date Time Formats
| Format | Meaning         | DataType | Sample Output              |
| ------ | --------------- | ------------ | ---------------------- |
| y      | year            | year         | 2020; 20               |
| D      | day-of-year     | number(3)    | 189                    |
| M/L    | month-of-year   | month        | 7; 07; Jul; July       |
| d      | day-of-month    | number(3)    | 28                     |
| Q/q    | quarter-of-year | number/text  | 3; 03; Q3; 3rd quarter |
| E      | day-of-week     | text         | Tue; Tuesday           |


In [0]:
storageAccountKey='ZRBZwKzOJm7ljZuuCEPCbTB/ITYEs8s8CPILkIP1kWBRgdYOHmhrKFA7yAXjFvBF6uwIg1LPTHq2+AStap+5UA=='
spark.conf.set("fs.azure.account.key.adldatalakehouseyubai.dfs.core.windows.net",storageAccountKey)

In [0]:
sourceCSVFilePath = 'abfss://working-labs@adldatalakehouseyubai.dfs.core.windows.net/bronze/daily-pricing/csv'
sourcePARQUETFilePath = 'abfss://working-labs@adldatalakehouseyubai.dfs.core.windows.net/bronze/daily-pricing/parquet'

In [0]:
sourcePARQUETFileDF = (spark
                       .read
                       .load(sourcePARQUETFilePath)
)

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
sourcePARQUETTransFileDF = (sourcePARQUETFileDF
 .withColumn("datalake_file_load_date", current_timestamp())
             )

In [0]:
display(sourcePARQUETTransFileDF)

In [0]:
from pyspark.sql.functions import year ,  quarter
(sourcePARQUETTransFileDF
.select("datalake_file_load_date")
.withColumn("datalake_file_load_date_year" , year("datalake_file_load_date"))
.show()
)

In [0]:
from pyspark.sql.functions import month , dayofmonth
(sourcePARQUETTransFileDF
.select("datalake_file_load_date")
.withColumn("datalake_file_load_date_year" , year("datalake_file_load_date"))
.withColumn("datalake_file_load_date_month" , month("datalake_file_load_date"))
.withColumn("datalake_file_load_date_day" , dayofmonth("datalake_file_load_date"))
.show()
)

In [0]:
from pyspark.sql.functions import concat 
(sourcePARQUETTransFileDF
.select("datalake_file_load_date")
.withColumn("dl_file_load_date_format" , concat(year("datalake_file_load_date"), month("datalake_file_load_date"),dayofmonth("datalake_file_load_date")))
.show()
)

In [0]:
from pyspark.sql.functions import date_format
(sourcePARQUETTransFileDF
.select("datalake_file_load_date")
.withColumn("dl_file_load_date_format" , date_format("datalake_file_load_date","yyyyMMdd"))
.show()
)

In [0]:
from pyspark.sql.functions import to_date
(sourcePARQUETFileDF
.select("DATE_OF_PRICING")
.withColumn("PRICING_DATE" , to_date("DATE_OF_PRICING","dd/MM/yyyy"))
.show()
)

In [0]:
(sourcePARQUETFileDF
.select("DATE_OF_PRICING")
.withColumn("PRICING_DATE" , to_date("DATE_OF_PRICING","dd/MM/yyyy"))
.withColumn("PRICING_DATE_FORMAT",date_format("PRICING_DATE","yyyyMMdd"))
.show()
)
